In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler,normalize
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import itertools
from sklearn.linear_model import ElasticNet

In [15]:
df_raw = pd.read_csv('Data\Measurment_dataset.csv')

In [ ]:
#decadal_average_dataset.csv is the dataset contains the estiamted emission rates from the 2010s to the 2090s
df = pd.read_csv('Data/decadal_average_dataset.csv')
# pipline_2015_2050_dataset contains the estimated total GHG emissions for planned hydropower reservoirs from the 2010s to 2050
#df = pd.read_csv('Data/pipline_2015_2050_dataset.csv')
# GHD_2015_2050_dataset contains the estimated total GHG emissions for constructed hydropower reservoirs from the 2010s to 2050
#df = pd.read_csv('Data/GHD_2015_2050_dataset.csv')
# GHD_2091_2100_dataset contains the estimated total GHG emissions for planned hydropower reservoirs from the 2091 to 2100
#df = pd.read_csv('Data/pipline_2091_2100_dataset.csv')
# GHD_2091_2100_dataset contains the estimated total GHG emissions for constructed hydropower reservoirs from the 2091 to 2100
#df = pd.read_csv('Data/GHD_2091_2100_dataset.csv')

### Modelling of CO2 Diffusive, CH4 diffusive and CH4 bubbling pathway

In [19]:
def process_dataframe(df, target_column, test_size=0.1, random_state=42):
    # List of all possible CH4 columns
    carbon_columns = ['CO2_diffusive__mgCO2_m_2_h_1_',
                   'CH4_diffusive__mgCH4_m_2_h_1_', 
                   'CH4_Bubbling__mgCH4_m_2_h_1_']

    # Drop the columns that are not the target column
    filtered_df = df.drop(columns=[col for col in carbon_columns if col != target_column])
    filtered_df = filtered_df.dropna()
    
    log_transform_columns = ['Volume_million_m3', 'Reservoir_area__km2_','CH4_Bubbling__mgCH4_m_2_h_1_']
    
    # Apply log transformation to specified columns
    for column in log_transform_columns:
        if column in filtered_df.columns:
            filtered_df[f'log_{column}'] = np.log10(filtered_df[column]) 
            filtered_df.drop(columns=[column], inplace=True)

    # Update the target column name if it was log-transformed
    if target_column in log_transform_columns:
        target_column = f'log_{target_column}'
    
    X = filtered_df.drop(columns=[target_column])
    y = filtered_df[target_column]
    
    # 划分数据集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    
    # 数据归一化
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

In [21]:
df_co2 = df_raw[['CO2_diffusive__mgCO2_m_2_h_1_', 'CH4_diffusive__mgCH4_m_2_h_1_','CH4_Bubbling__mgCH4_m_2_h_1_', 'Temperature____', 'Precipitation_kg_m_2_month', 'VPD_Pa', 'rsds_MJ_m2_d', 'Reservoir_area__km2_']]
CO2_X_train_scaled, CO2_X_test_scaled, CO2_y_train, CO2_y_test, CO2_scaler = process_dataframe(df_co2, 'CO2_diffusive__mgCO2_m_2_h_1_')
df_ch4 = df_raw[['CO2_diffusive__mgCO2_m_2_h_1_', 'CH4_diffusive__mgCH4_m_2_h_1_', 'CH4_Bubbling__mgCH4_m_2_h_1_', 'Wind_speed__m_s_1_', 'Age', 'Precipitation_kg_m_2_month', 'VPD_Pa', 'NPP_g_C_m_2_yr', 'Volume_million_m3']]
CH4_X_train_scaled, CH4_X_test_scaled, CH4_y_train, CH4_y_test, CH4_scaler = process_dataframe(df_ch4, 'CH4_diffusive__mgCH4_m_2_h_1_')
df_ebu_ch4 = df_raw[['CO2_diffusive__mgCO2_m_2_h_1_', 'CH4_diffusive__mgCH4_m_2_h_1_', 'CH4_Bubbling__mgCH4_m_2_h_1_', 'Age','VPD_Pa', 'HURS_%', 'rsds_MJ_m2_d']]
ebu_CH4_X_train_scaled, ebu_CH4_X_test_scaled, ebu_CH4_y_train, ebu_CH4_y_test,ebu_CH4_scaler = process_dataframe(df_ebu_ch4, 'CH4_Bubbling__mgCH4_m_2_h_1_')

### Modelling of CH4 degassing pathway

In [69]:
def elastic_grid_search(X_train_scaled, X_test_scaled, y_train, y_test, carbon_columns):
    param_grid = {
        'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
        'l1_ratio': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    }

    elastic_net = ElasticNet()
    grid_search = GridSearchCV(estimator=elastic_net, param_grid=param_grid, cv=5, scoring='r2')
    grid_search.fit(X_train_scaled, y_train)

    print("Best alpha:", grid_search.best_params_['alpha'])
    print("Best l1_ratio:", grid_search.best_params_['l1_ratio'])

    best_elastic_net = grid_search.best_estimator_
    best_elastic_net.fit(X_train_scaled, y_train)

    # Predictions (still in log10 space)
    y_pred = best_elastic_net.predict(X_test_scaled)
    y_train_pred = best_elastic_net.predict(X_train_scaled)

    print("Training set")
    print(f"R^2: {r2_score(y_train, y_train_pred)}")
    print("Testing set")
    print(f"R^2: {r2_score(y_test, y_pred)}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred))}")
    print(carbon_columns)
    coefficients = best_elastic_net.coef_
    print("Coefficients of the ElasticNet model:", coefficients)
    print("Intercept of the ElasticNet model:", best_elastic_net.intercept_)
    return best_elastic_net

In [71]:
deg_target_column = 'Difference CH4 concentration reservoir intake-outlet (mgC L-1)'
deg_filtered_df  = df_raw[['Mean_depth__m_','Temperature____', 'Wind_speed__m_s_1_','Age','Precipitation_kg_m_2_month',
    'discharge__m3_s_1_', 'VPD_Pa','HURS_%','NPP_g_C_m_2_yr','rsds_MJ_m2_d','Volume_million_m3','Reservoir_area__km2_','Difference CH4 concentration reservoir intake-outlet (mgC L-1)']].dropna()
# Columns to log transform
deg_log_transform_columns = ['Volume_million_m3','Reservoir_area__km2_']

# Apply log transformation to specified columns
for column in deg_log_transform_columns:
    if column in deg_filtered_df.columns:
        deg_filtered_df[f'log_{column}'] = np.log10(deg_filtered_df[column]) 
        deg_filtered_df.drop(columns=[column], inplace=True)

deg_X = deg_filtered_df.drop(columns=[deg_target_column])
print(deg_X.columns)
deg_y = deg_filtered_df[deg_target_column]
deg_X_train, deg_X_test, deg_y_train, deg_y_test = train_test_split(deg_X, deg_y, test_size=0.1, random_state=42)
#deg_label_df = df_raw.copy()
#deg_train_indices = deg_X_train.index
#deg_test_indices = deg_X_test.index
#deg_label_df['split'] = 'train'  # Default to 'train'
#deg_label_df.loc[deg_test_indices, 'split'] = 'test'

deg_scaler = MinMaxScaler(feature_range=(0, 1))
deg_X_train_scaled = deg_scaler.fit_transform(deg_X_train)
deg_X_test_scaled = deg_scaler.transform(deg_X_test)
deg_elastic_net = elastic_grid_search(deg_X_train_scaled, deg_X_test_scaled, deg_y_train, deg_y_test, 'CH4_degassing__mgC_L_')

Index(['Mean_depth__m_', 'Temperature____', 'Wind_speed__m_s_1_', 'Age',
       'Precipitation_kg_m_2_month', 'discharge__m3_s_1_', 'VPD_Pa', 'HURS_%',
       'NPP_g_C_m_2_yr', 'rsds_MJ_m2_d', 'log_Volume_million_m3',
       'log_Reservoir_area__km2_'],
      dtype='object')
Best alpha: 0.01
Best l1_ratio: 1.0
Training set
R^2: 0.8040998131952781
Testing set
R^2: 0.6342544554920044
CH4_degassing__mgC_L_
Coefficients of the ElasticNet model: [-0.95699823 -0.         -1.04516335 -0.          3.01399414 -0.43336516
  0.85013955  0.22341784 -0.11780494  0.          1.04048779  0.        ]
Intercept of the ElasticNet model: -3.843182244415281


### Emission rates estimation

In [25]:
def prediction_func (df, model, X_train_scaled,y_train,scaler):
    df_clean = df.dropna()
    log_transform_columns = ['Volume_million_m3','Reservoir_area__km2_']
 
    # Apply log transformation to specified columns
    for column in log_transform_columns:
        if column in df_clean.columns:
            df_clean[f'log_{column}'] = np.log10(df_clean[column])
            df_clean.drop(columns=[column], inplace=True)
    df_scaled = scaler.transform(df_clean)
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(df_scaled)
    return predictions, df_clean

In [77]:
df_co2_pred = df[['Temperature____', 'Precipitation_kg_m_2_month', 'VPD_Pa', 'rsds_MJ_m2_d', 'Reservoir_area__km2_']].dropna()
#df_co2_pred.rename(columns={'Area (km²)': 'Reservoir_area__km2_'}, inplace=True)
CO2_model = XGBRegressor(random_state=42, learning_rate=0.01, max_depth = 8, n_estimators= 200, subsample =0.6)
co2_pred, df_co2_pred_clean= prediction_func(df_co2_pred,CO2_model,CO2_X_train_scaled,CO2_y_train,CO2_scaler,False,True)
# Step 4: Create a new column in the original df2 and fill it with NaN
df['CO2_mg_m2_h'] = np.nan
# Step 5: Assign the predictions back to the corresponding rows in the original df2
df.loc[df_co2_pred_clean.index, 'CO2_mg_m2_h'] = co2_pred
df['CO2_gCO2_m2_yr']=df['CO2_mg_m2_h']*24*365/1000

In [75]:
df_ch4_pred = df[['Wind_speed__m_s_1_', 'Age', 'Precipitation_kg_m_2_month', 'VPD_Pa', 'NPP_g_C_m_2_yr', 'modelled_storage_million_m3']].dropna()
df_ch4_pred.rename(columns={'modelled_storage_million_m3': 'Volume_million_m3'}, inplace=True)
CH4_model = SVR(C=100, epsilon=0.01, kernel='rbf')
ch4_pred, df_ch4_pred_clean= prediction_func(df_ch4_pred,CH4_model,CH4_X_train_scaled,CH4_y_train,CH4_scaler,False,True)
# Step 4: Create a new column in the original df2 and fill it with NaN
df['Ch4_mgch4_m2_h'] = np.nan
# Step 5: Assign the predictions back to the corresponding rows in the original df2
df.loc[df_ch4_pred_clean.index, 'Ch4_mgch4_m2_h'] = ch4_pred
df['Ch4_gCO2_m2_yr']=df['Ch4_mgch4_m2_h']*24*365/1000*34

In [77]:
df_ebu_ch4_pred = df[['Age','VPD_Pa','HURS_%', 'rsds_MJ_m2_d']].dropna()
#df_ebu_ch4_pred.rename(columns={'Area (km²)': 'Reservoir_area__km2_','modelled_storage_million_m3': 'Volume_million_m3'}, inplace=True)
ebu_CH4_model = SVR(C=10, epsilon=0.01, kernel='rbf')
ebu_ch4_pred, df_ebu_ch4_pred_clean= prediction_func(df_ebu_ch4_pred,ebu_CH4_model,ebu_CH4_X_train_scaled, ebu_CH4_y_train,ebu_CH4_scaler)
# Step 4: Create a new column in the original df2 and fill it with NaN
df['ebu_Ch4_mgch4_m2_h'] = np.nan
# Step 5: Assign the predictions back to the corresponding rows in the original df2
df.loc[df_ebu_ch4_pred_clean.index, 'ebu_Ch4_mgch4_m2_h'] = 10**ebu_ch4_pred
df['ebu_Ch4_gCO2_m2_yr']=df['ebu_Ch4_mgch4_m2_h']*24*365/1000*34

In [81]:
#df_deg_ch4_pred = df[['Mean_depth__m_','Temperature____', 'Wind_speed__m_s_1_','Age',
      # 'Precipitation_kg_m_2_month', 'discharge__m3_s_1_', 'VPD_Pa', 'HURS_%',
       #'NPP_g_C_m_2_yr','rsds_MJ_m2_d','Reservoir_area__km2_','Volume_million_m3']].dropna()
df_deg_ch4_pred = df[['Depth (m)','Temperature____', 'Wind_speed__m_s_1_','Age',
       'Precipitation_kg_m_2_month', 'discharge__m3_s_1_', 'VPD_Pa', 'HURS_%',
       'NPP_g_C_m_2_yr','rsds_MJ_m2_d','Area (km²)','modelled_storage_million_m3']].dropna()
df_deg_ch4_pred.rename(columns={'Depth (m)':'Mean_depth__m_', 'Area (km²)':'Reservoir_area__km2_','modelled_storage_million_m3': 'Volume_million_m3'}, inplace=True)
deg_ch4_pred, df_deg_ch4_pred_clean= prediction_func(df_deg_ch4_pred,deg_elastic_net,deg_X_train_scaled, deg_y_train,deg_scaler, False, True)
# Step 4: Create a new column in the original df2 and fill it with NaN
df['deg_Ch4_mgC_L'] = np.nan
# Step 5: Assign the predictions back to the corresponding rows in the original df2
df.loc[df_deg_ch4_pred_clean.index, 'deg_Ch4_mgC_L'] = deg_ch4_pred
df['deg_gco2_m_2_yr'] = df['deg_Ch4_mgC_L']*df['discharge__m3_s_1_']/df['Reservoir_area__km2_']*1000*60*60*24*365/1000/12*16*34/1000000

### Total GHG emissions

In [25]:
df['total_emissions_ghg_rate_gco2_m_2_yr'] = df['CO2_gCO2_m2_yr']+df['Ch4_gCO2_m2_yr']+df['ebu_Ch4_gCO2_m2_yr']+df['deg_gco2_m_2_yr']

In [27]:
df['total_emissions_Gg_yr'] = df['total_emissions_ghg_rate_gco2_m_2_yr']*df['Reservoir_area__km2_']/1000